# Assignment 7: MultiIndex DataFrame

In [1]:
import numpy as np
import pandas as pd


def make_multiindex_df(rng):
    index = pd.MultiIndex.from_product([["A", "B"], [1, 2, 3]], names=["Category", "Number"])
    values = rng.integers(1, 100, size=len(index))
    return pd.DataFrame({"Value": values}, index=index)


def get_category_slice(df, category):
    return df.loc[category]


def get_specific(df, category, number):
    return df.loc[(category, number)]


def make_category_subcategory_df(rng, n=10):
    categories = rng.choice(["Cat1", "Cat2"], size=n)
    subcats = rng.choice(["Sub1", "Sub2"], size=n)
    values = rng.integers(1, 50, size=n)
    df = pd.DataFrame({"Category": categories, "SubCategory": subcats, "Value": values})
    return df.set_index(["Category", "SubCategory"])


def sum_by_category_subcategory(df):
    return df.groupby(level=["Category", "SubCategory"])["Value"].sum()

In [2]:
# sample answer
rng = np.random.default_rng(42)
df1 = make_multiindex_df(rng)
print(df1)
print(get_category_slice(df1, "A"))
print(get_specific(df1, "A", 2))

df2 = make_category_subcategory_df(rng)
print(df2)
print(sum_by_category_subcategory(df2))

                 Value
Category Number       
A        1           9
         2          77
         3          65
B        1          44
         2          43
         3          86
        Value
Number       
1           9
2          77
3          65
Value    77
Name: (A, 2), dtype: int64
                      Value
Category SubCategory       
Cat1     Sub2            20
Cat2     Sub1            41
Cat1     Sub2            27
         Sub1            22
Cat2     Sub2            23
         Sub1            12
         Sub1             5
         Sub2            28
         Sub2            44
         Sub2             4
Category  SubCategory
Cat1      Sub1           22
          Sub2           47
Cat2      Sub1           58
          Sub2           99
Name: Value, dtype: int64


In [3]:
# edge and negative case checks
df = make_multiindex_df(np.random.default_rng(7))
sub = get_category_slice(df, "A")
assert list(sub.index) == [1, 2, 3]
val = get_specific(df, "A", 2)
assert val["Value"] == df.loc[("A", 2), "Value"]

sub_b = get_category_slice(df, "B")
assert list(sub_b.index) == [1, 2, 3]

try:
    get_category_slice(df, "C")
    assert False, "expected KeyError for a missing category"
except KeyError:
    pass

data = pd.DataFrame({
    "Category": ["Cat1", "Cat1", "Cat2", "Cat2", "Cat1"],
    "SubCategory": ["Sub1", "Sub2", "Sub1", "Sub1", "Sub1"],
    "Value": [10, 20, 30, 40, 50],
}).set_index(["Category", "SubCategory"])
result = sum_by_category_subcategory(data)
assert result.loc[("Cat1", "Sub1")] == 60
assert result.loc[("Cat1", "Sub2")] == 20
assert result.loc[("Cat2", "Sub1")] == 70

single_row = pd.DataFrame({
    "Category": ["Cat1"],
    "SubCategory": ["Sub1"],
    "Value": [99],
}).set_index(["Category", "SubCategory"])
assert sum_by_category_subcategory(single_row).loc[("Cat1", "Sub1")] == 99

raised = False
try:
    data.groupby(level=["Category", "NotALevel"])
except Exception:
    raised = True
assert raised, "expected an error for an unknown level name"

print("all tests passed")

all tests passed
